In [1]:
import warnings
import pandas as pd
import os


# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None,
                    percentage_change=None, time_limit_minutes=None, ignore_time_interval_before=None, ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        if adjusted_signal_datetime not in price_data.index:
            continue

        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']
        
        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 3:
                result = 'Ignored'
                reason = 'More than 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 2:
                if later_exits[-1][1] == side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'Two open trades, last one with different side'
                    ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # New logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
            if ignore_signal:
                new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Signal Open Price': signal_open_price,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Ignored',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00',
                    'ROI': 0,
                    'NAV': current_margin,
                    'Ignore Reason': 'Signal around economic event'
                }])
                output_data = pd.concat([output_data, new_row], ignore_index=True)
                continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, time_limit_minutes, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        # Check for economic data event before the trade exit
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                    else:
                        exit_price = price_data.iloc[price_data.index.get_loc(exit_datetime, method='nearest')]['Open']
                        result = 'ended before data with no exact price'
                    break

        if result not in ['ended before data with profit', 'ended before data with loss', 'ended before data with no exact price']:
            if result == 1:
                current_margin = current_margin * (1 + tp)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
    
    return output_data

# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

def calculate_metrics(group, initial_nav):
    total_trades = len(group[(group['Result'] == 1) | (group['Result'] == -1)])
    total_wins = len(group[group['Result'] == 1])
    total_losses = len(group[group['Result'] == -1])
    win_rate = total_wins / total_trades if total_trades > 0 else 0
    
    final_nav = group['NAV'].iloc[-1] if total_trades > 0 else initial_nav
    roi = ((final_nav - initial_nav) / initial_nav) * 100
    
    drawdown = 0
    cumulative_returns = (group['NAV'] - initial_nav).cumsum()
    peak = cumulative_returns.cummax()
    drawdown = (peak - cumulative_returns).max()
    
    return {
        'Total Trades': total_trades,
        'Total Wins': total_wins,
        'Total Losses': total_losses,
        'Win Rate': win_rate,
        'ROI': roi,
        'NAV': final_nav,
        'Max Drawdown': drawdown
    }

def generate_report(price_data, signal_data, scenarios, output_directory, output_file_name):
    report_columns = ['Scenario', 'Period', 'Total Trades', 'Total Wins', 'Total Losses', 'Win Rate', 'ROI', 'NAV', 'Max Drawdown']
    report_data = pd.DataFrame(columns=report_columns)

    for scenario in scenarios:
        tp = scenario['tp']
        sl = scenario['sl']
        percentage_change = scenario.get('percentage_change', None)
        entry_time_offset = scenario.get('entry_time_offset', None)
        time_limit_minutes = scenario.get('time_limit_minutes', None)
        ignore_time_interval_before = scenario.get('ignore_time_interval_before', None)
        ignore_time_interval_after = scenario.get('ignore_time_interval_after', None)
        
        # Backtest the trades for the current scenario
        trade_data = backtest_trades(price_data, signal_data, tp, sl, entry_time_offset, percentage_change, time_limit_minutes, ignore_time_interval_before, ignore_time_interval_after)
        monthly_groups = trade_data.groupby(trade_data['Datetime'].dt.to_period('M'))
        monthly_reports = []
        initial_nav = 100000
        for month, group in monthly_groups:
            metrics = calculate_metrics(group, initial_nav)
            metrics['Period'] = month.strftime('%Y-%m')
            metrics['Scenario'] = f"TP={tp}, SL={sl}, Offset={entry_time_offset}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}, IgnoreBefore={ignore_time_interval_before}, IgnoreAfter={ignore_time_interval_after}"
            monthly_reports.append(pd.DataFrame([metrics]))
            initial_nav = metrics['NAV']

        if monthly_reports:
            monthly_report = pd.concat(monthly_reports, ignore_index=True)
            report_data = pd.concat([report_data, monthly_report], ignore_index=True)

        overall_metrics = calculate_metrics(trade_data, 100000)
        overall_metrics['Period'] = 'Overall'
        overall_metrics['Scenario'] = f"TP={tp}, SL={sl}, Offset={entry_time_offset}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}, IgnoreBefore={ignore_time_interval_before}, IgnoreAfter={ignore_time_interval_after}"
        overall_report = pd.DataFrame([overall_metrics])
        report_data = pd.concat([report_data, overall_report], ignore_index=True)

    os.makedirs(output_directory, exist_ok=True)
    report_data.to_csv(os.path.join(output_directory, output_file_name), index=False)

    return report_data



In [2]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\\Updated_2024_Signal_Dataset_with_events.csv', parse_dates=['Datetime'])

In [ ]:
# # Define the parameters
# tp = 0.014
# sl = 0.014
# entry_time_offset = 110  # Time offset in minutes
# percentage_change = 0.0001
# time_limit_minutes = 120
# 
# # Call the backtest_trades function
# backtest_output = backtest_trades(
#     price_data=price_data,
#     signal_data=signal_data,
#     tp=tp,
#     sl=sl,
#     entry_time_offset=entry_time_offset,
#     percentage_change=percentage_change,
#     time_limit_minutes=time_limit_minutes,
#     ignore_time_interval_before=1080,
#     ignore_time_interval_after=0
# )
# 
# backtest_output

In [3]:
 # Define your scenarios
scenarios = [
      {'sl': 0.012, 'entry_time_offset': 60, 'time_limit_minutes':120}
  ]
  
# Add ignore time intervals to the scenarios
ignore_intervals_before = [0,30,60,60*2,60*3,60*4,60*5,60*6,60*6,60*7,60*8,60*9,60*10,60*11,60*12,60*13,60*14,60*15,60*16,60*17,60*18,60*19,60*20]
ignore_intervals_after = [0]
pct_intervals = [0.0002]
tp_intervals = [0.014]
extended_scenarios = []
  
for scenario in scenarios:
    for interval_before in ignore_intervals_before:
        for interval_after in ignore_intervals_after:
            for pct_interval in pct_intervals:
                 for tp_interval in tp_intervals:
                   extended_scenario = scenario.copy()
                   extended_scenario['ignore_time_interval_before'] = interval_before
                   extended_scenario['ignore_time_interval_after'] = interval_after
                   extended_scenario['percentage_change'] = pct_interval
                   extended_scenario['tp'] = tp_interval
                   extended_scenarios.append(extended_scenario)
  
  
# Call the generate_report function with the extended scenarios
output_directory = 'E:\Signal Backtesting\Output'
output_file_name = 'macro_june_analysis.csv'
  
report = generate_report(price_data, signal_data, extended_scenarios, output_directory, output_file_name)

In [15]:
# Define your scenarios
scenarios = [
    {'tp':0.014,'sl': 0.012, 'entry_time_offset': 60, 'percentage_change':0.0002 ,'time_limit_minutes':120, 'ignore_time_interval_before':1080, 'ignore_time_interval_after':0}
]
# Call the generate_report function with the extended scenarios
output_directory = 'E:\Signal Backtesting\Output'
output_file_name = 'optimized_June_analysis.csv'

report = generate_report(price_data, signal_data, scenarios, output_directory, output_file_name)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.014, SL=0.012, Offset=60, pctChange=0.000...",2024-01,26,13,13,0.500000,2.407955,102407.955205,6101.430824
1,"TP=0.014, SL=0.012, Offset=60, pctChange=0.000...",2024-02,16,8,8,0.500000,1.475032,103918.505252,671.795989
2,"TP=0.014, SL=0.012, Offset=60, pctChange=0.000...",2024-03,31,16,15,0.516129,4.222491,108306.454742,48989.164358
3,"TP=0.014, SL=0.012, Offset=60, pctChange=0.000...",2024-04,33,19,14,0.575758,9.981179,119116.715354,0.000000
4,"TP=0.014, SL=0.012, Offset=60, pctChange=0.000...",2024-05,26,13,13,0.500000,2.407955,121984.992502,20318.578694
5,"TP=0.014, SL=0.012, Offset=60, pctChange=0.000...",2024-06,25,16,9,0.640000,12.052044,136686.676877,3983.038044
6,"TP=0.014, SL=0.012, Offset=60, pctChange=0.000...",Overall,157,85,72,0.541401,36.686677,136686.676877,6101.430824


In [18]:
# Define your scenarios
scenarios = [
    {'tp':0.013,'sl': 0.014, 'entry_time_offset': 60, 'percentage_change':0.0006 ,'time_limit_minutes':120, 'ignore_time_interval_before':1080, 'ignore_time_interval_after':0}
]
# Call the generate_report function with the extended scenarios
output_directory = 'E:\Signal Backtesting\Output'
output_file_name = 'optimized_May_analysis.csv'

report = generate_report(price_data, signal_data, scenarios, output_directory, output_file_name)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.013, SL=0.014, Offset=60, pctChange=0.000...",2024-01,24,13,11,0.541667,1.290532,101290.532440,4907.405488
1,"TP=0.013, SL=0.014, Offset=60, pctChange=0.000...",2024-02,15,9,6,0.600000,3.215875,104547.909180,1076.255547
2,"TP=0.013, SL=0.014, Offset=60, pctChange=0.000...",2024-03,30,17,13,0.566667,3.695633,108411.615790,76323.437549
3,"TP=0.013, SL=0.014, Offset=60, pctChange=0.000...",2024-04,31,20,11,0.645161,10.875342,120201.749530,256.133595
4,"TP=0.013, SL=0.014, Offset=60, pctChange=0.000...",2024-05,24,17,7,0.708333,12.849328,135646.866104,142.078468
5,"TP=0.013, SL=0.014, Offset=60, pctChange=0.000...",2024-06,22,15,7,0.681818,9.971484,149172.871432,1280.404372
6,"TP=0.013, SL=0.014, Offset=60, pctChange=0.000...",Overall,146,91,55,0.623288,49.172871,149172.871432,4907.405488


In [19]:
# Define your scenarios
scenarios = [
    {'tp':0.012,'sl': 0.012, 'entry_time_offset': 70, 'percentage_change':0.0007 ,'time_limit_minutes':120, 'ignore_time_interval_before':1080, 'ignore_time_interval_after':0}
]
# Call the generate_report function with the extended scenarios
output_directory = 'E:\Signal Backtesting\Output'
output_file_name = 'optimized_Apr_analysis.csv'

report = generate_report(price_data, signal_data, scenarios, output_directory, output_file_name)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.012, SL=0.012, Offset=70, pctChange=0.000...",2024-01,26,17,9,0.653846,9.870529,109870.528572,28.800000
1,"TP=0.012, SL=0.012, Offset=70, pctChange=0.000...",2024-02,17,10,7,0.588235,3.538945,113758.786591,0.000000
2,"TP=0.012, SL=0.012, Offset=70, pctChange=0.000...",2024-03,30,15,15,0.500000,-0.215782,113513.315142,79450.947297
3,"TP=0.012, SL=0.012, Offset=70, pctChange=0.000...",2024-04,27,21,6,0.777778,19.490239,135637.331659,0.000000
4,"TP=0.012, SL=0.012, Offset=70, pctChange=0.000...",2024-05,24,14,10,0.583333,4.736154,142061.324152,12186.629506
5,"TP=0.012, SL=0.012, Offset=70, pctChange=0.000...",2024-06,23,15,8,0.652174,8.583352,154254.947636,122.732147
6,"TP=0.012, SL=0.012, Offset=70, pctChange=0.000...",Overall,147,92,55,0.625850,54.254948,154254.947636,28.800000


In [20]:
# Define your scenarios
scenarios = [
    {'tp':0.009,'sl': 0.014, 'entry_time_offset': 110, 'percentage_change':0.0006 ,'time_limit_minutes':120, 'ignore_time_interval_before':1080, 'ignore_time_interval_after':0}
]
# Call the generate_report function with the extended scenarios
output_directory = 'E:\Signal Backtesting\Output'
output_file_name = 'optimized_Mar_analysis.csv'

report = generate_report(price_data, signal_data, scenarios, output_directory, output_file_name)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.009, SL=0.014, Offset=110, pctChange=0.00...",2024-01,19,17,2,0.894737,13.214918,113214.917887,0.000000
1,"TP=0.009, SL=0.014, Offset=110, pctChange=0.00...",2024-02,13,8,5,0.615385,0.118401,113348.965421,1753.987566
2,"TP=0.009, SL=0.014, Offset=110, pctChange=0.00...",2024-03,32,24,8,0.750000,10.765193,125551.200502,0.000000
3,"TP=0.009, SL=0.014, Offset=110, pctChange=0.00...",2024-04,29,19,10,0.655172,2.967770,129277.271573,0.000000
4,"TP=0.009, SL=0.014, Offset=110, pctChange=0.00...",2024-05,26,17,9,0.653846,2.575126,132606.324628,6980.908509
5,"TP=0.009, SL=0.014, Offset=110, pctChange=0.00...",2024-06,22,16,6,0.727273,6.052314,140632.075917,2718.960080
6,"TP=0.009, SL=0.014, Offset=110, pctChange=0.00...",Overall,141,101,40,0.716312,40.632076,140632.075917,0.000000


In [21]:
# Define your scenarios
scenarios = [
    {'tp':0.013,'sl': 0.01, 'entry_time_offset': 70, 'percentage_change':0.0005 ,'time_limit_minutes':120, 'ignore_time_interval_before':1080, 'ignore_time_interval_after':0}
]
# Call the generate_report function with the extended scenarios
output_directory = 'E:\Signal Backtesting\Output'
output_file_name = 'optimized_Feb_analysis.csv'

report = generate_report(price_data, signal_data, scenarios, output_directory, output_file_name)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.013, SL=0.01, Offset=70, pctChange=0.0005...",2024-01,27,14,13,0.518519,5.145276,105145.275872,4287.245847
1,"TP=0.013, SL=0.01, Offset=70, pctChange=0.0005...",2024-02,18,12,6,0.666667,9.932097,115588.406533,0.000000
2,"TP=0.013, SL=0.01, Offset=70, pctChange=0.0005...",2024-03,32,15,17,0.468750,2.315160,118264.463311,12537.736104
3,"TP=0.013, SL=0.01, Offset=70, pctChange=0.0005...",2024-04,29,15,14,0.517241,5.447043,124706.379261,509.630603
4,"TP=0.013, SL=0.01, Offset=70, pctChange=0.0005...",2024-05,24,10,14,0.416667,-1.147609,123275.237532,28155.455547
5,"TP=0.013, SL=0.01, Offset=70, pctChange=0.0005...",2024-06,25,13,12,0.520000,4.844373,129247.149266,14370.199193
6,"TP=0.013, SL=0.01, Offset=70, pctChange=0.0005...",Overall,155,79,76,0.509677,29.247149,129247.149266,4287.245847


In [22]:
# Define your scenarios
scenarios = [
    {'tp':0.009,'sl': 0.014, 'entry_time_offset': 140, 'percentage_change':0.0004 ,'time_limit_minutes':120, 'ignore_time_interval_before':1080, 'ignore_time_interval_after':0}
]
# Call the generate_report function with the extended scenarios
output_directory = 'E:\Signal Backtesting\Output'
output_file_name = 'optimized_Jan_analysis.csv'

report = generate_report(price_data, signal_data, scenarios, output_directory, output_file_name)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.009, SL=0.014, Offset=140, pctChange=0.00...",2024-01,27,24,3,0.888889,18.855350,118855.349792,0.000000
1,"TP=0.009, SL=0.014, Offset=140, pctChange=0.00...",2024-02,15,8,7,0.533333,-2.665291,115687.508763,41177.368191
2,"TP=0.009, SL=0.014, Offset=140, pctChange=0.00...",2024-03,31,22,9,0.709677,7.274844,124103.594562,28411.665355
3,"TP=0.009, SL=0.014, Offset=140, pctChange=0.00...",2024-04,30,19,11,0.633333,1.526221,125997.690160,12418.398102
4,"TP=0.009, SL=0.014, Offset=140, pctChange=0.00...",2024-05,26,17,9,0.653846,2.575126,129242.289851,0.000000
5,"TP=0.009, SL=0.014, Offset=140, pctChange=0.00...",2024-06,22,14,8,0.636364,1.272527,130886.932888,1324.991956
6,"TP=0.009, SL=0.014, Offset=140, pctChange=0.00...",Overall,151,104,47,0.688742,30.886933,130886.932888,0.000000
